In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from birddog.database import Database
from birddog.wiki import (
    page_label,
    sequential_page_label,
)
from birddog.nocodb_database import (
    clone_table_schema,
    copy_records,
    rename_field,
)

2026-04-12 14:51:45,019 [INFO] Using local nocodb api: http://localhost:8080
2026-04-12 14:51:45,176 [INFO] Translation is enabled. Using GCP translator
2026-04-12 14:51:45,176 [INFO] Using Google Cloud translation API
2026-04-12 14:51:45,177 [INFO] GoogleCloudTranslator using REST API


In [3]:
db = Database()

2026-04-12 14:51:48,060 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         20.00    42.22    39.00       0.00           24


In [ ]:
page_ids = db.get_all_ids("Pages")

In [ ]:
len(page_ids)

In [ ]:
db.delete("Pages", page_ids)

In [ ]:
doc_ids = db.get_all_ids("Documents")

In [ ]:
db.delete("Documents", doc_ids)

In [4]:
nocodb_aws_host = os.environ["BIRDDOG_AWS_NOCODB_HOST"]
nocodb_aws_token = os.environ["BIRDDOG_AWS_NOCODB_API_TOKEN"]
nocodb_aws_base_id = os.environ["BIRDDOG_AWS_BASE_ID"]

In [5]:
db2 = Database(host=nocodb_aws_host, api_token=nocodb_aws_token, base_id=nocodb_aws_base_id)
db2._host

'http://nocodb-env.eba-xhmfyydr.us-east-2.elasticbeanstalk.com'

In [6]:
schema, _ = db2.scan("Schema", limit=500) 

In [7]:
len(schema)

50

In [9]:
copy_records(db2, "Schema", db, "Schema")

2026-04-12 14:54:29,599 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         21.00     0.02    39.00       0.00           24
  nocodb.internal:api                   22.00     0.07    39.00       0.00           24


In [10]:
copy_records(db2, "Schema Values", db, "Schema Values")

2026-04-12 14:55:55,883 [INFO] 
AdaptiveThrottle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  localhost:api                         22.00     0.06    37.56       0.00           24
  nocodb.internal:api                   23.00     0.01    39.00       0.00           24


In [ ]:
pages = []
cursor = None
while True:
    print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=10000,
        where=("source_type", "neq", "wiki"),
        fields="Id")
    if batch:
        pages.extend([b["Id"] for b in batch])
    if not cursor:
        break

In [ ]:
len(pages)

In [ ]:
#db.delete("Pages", pages)

In [ ]:
docs = []
cursor = None
while True:
    print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Documents", 
        cursor=cursor, 
        limit=1000,
        where=("owning_pages", "eq", 0),
        fields="Id",
    )
    if batch:
        docs.extend([b["Id"] for b in batch])
    if not cursor:
        break
    #break

In [ ]:
docs[0]

In [ ]:
len(docs)

In [ ]:
#db.delete("Documents", docs)

In [ ]:
pages = []
cursor = None
while True:
    if cursor and (int(cursor) % 10000) == 0:
        print(f"cursor={cursor}")
    batch, cursor = db.scan(
        "Pages", 
        cursor=cursor, 
        limit=1000,
        where=("seq_label", "is", None),
        fields=("Id","title","label","seq_label"),
    )
    if batch:
        pages.extend(batch)
    if not cursor:
        break

In [ ]:
len(pages)

In [ ]:
pages[1000]

In [ ]:
def normalize_labels(page):
    result = page.copy()
    title = page.get("title")
    if title:
        proper_label = page_label(title)
        result["label"] = proper_label
        proper_seq_label = sequential_page_label(proper_label)
        result["seq_label"] = proper_seq_label
    return result

In [ ]:
normalize_labels(pages[2000])

In [ ]:
pages[2000]

In [ ]:
pages[2000] == normalize_labels(pages[2000])

In [ ]:
pages[2000] == pages[2000].copy()

In [ ]:
norm_pages = [normalize_labels(p) for p in pages]

In [ ]:
norm_pages[:10]

In [ ]:
changed_pages = [n for n,p in zip(norm_pages, pages) if n != p]

In [ ]:
len(changed_pages)

In [ ]:
len(pages)

In [ ]:
len(norm_pages)

In [ ]:
rec_ids = db.write("Pages", norm_pages[1000:2000])

In [ ]:
chunk = 1000
for i in range(0, len(norm_pages), chunk):
    print(i)
    rec_ids = db.write("Pages", norm_pages[i:(i+chunk)])